In [1]:
import argparse
from config import M3E_embeddings_model_path, BGE_embeddings_model_path, SimModel_path
from generate_answer import question_test
from test_score import test_metrics

In [3]:
# 使用modelscope下载模型
from modelscope import snapshot_download
snapshot_download('qwen/Qwen2.5-7B-Instruct', local_dir='./models/Qwen2.5-7B-Instruct')


2026-04-03 17:57:03,580 - modelscope - INFO - Download model 'qwen/Qwen2.5-7B-Instruct' successfully.
2026-04-03 17:57:03,582 - modelscope - INFO - Target directory already exists, skipping creation.


'./models/Qwen2.5-7B-Instruct'

In [ ]:
# 使用transformers加载模型
from transformers import AutoTokenizer, AutoModelForCausalLM
model_name = "Qwen2.5-7B-Instruct"
cache_dir = "./models/Qwen2.5-7B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    cache_dir=cache_dir
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    cache_dir=cache_dir
)

In [2]:
parser = argparse.ArgumentParser(description='Intelligent Cabin Automotive Knowledge Q&A System')
# 选择大语言模型
parser.add_argument('--llm_name', default='qwen2', type=str,
                    choices=['qwen2', 'baichuan2', 'chatglm3', "Qwen/Qwen3.5-9B", "Qwen/Qwen3-9B"],
                    help='Select the Large Language Model for Generating Responses')
# 选择重排序模型
parser.add_argument('--reranker_name', default='bce', type=str, choices=['bce', 'bge'],
                    help='Select the reranking model used for reordering the retrieved documents')
# 选择文本嵌入模型
parser.add_argument('--m3e_embeddings_model', default=M3E_embeddings_model_path, type=str,
                    help='The path to the text embedding model for recall used by M3E')
# 选择文本嵌入模型
parser.add_argument('--bge_embeddings_model', default=BGE_embeddings_model_path, type=str,
                    help='The path to the text embedding model for recall used by BGE')
# 选择是否进行提示词优化
parser.add_argument('--prompt_enhance', default=True, type=str,
                    help='Choose to optimize the prompt')
# 选择单路径召回的最大文本长度
parser.add_argument('--single_max_length', default=4000, type=int,
                    help='The maximum text length for single-path recall')
# 选择单路径召回的最大召回数量
parser.add_argument('--single_top_k', default=6, type=int,
                    help='The maximum number of retrievals for single-path recall')
# 选择多路径召回的最大文本长度
parser.add_argument('--mutil_max_length', default=4000, type=int,
                    help='The maximum text length for multi-path recall')
# 选择多路径召回的最大召回数量
parser.add_argument('--mutil_top_k', default=6, type=int,
                    help='The maximum number of retrievals for multi-path recall')
# 选择PDF文件的路径
parser.add_argument('--pdf_path', default="./data/car_user_manual.pdf", type=str,
                    help='The path to the PDF file')
# 选择测试集的路径
parser.add_argument('--test_path', default="./data/test_question.json", type=str,
                    help='The path to the test dataset')
# 选择预测结果的存储路径
parser.add_argument('--predict_path', default="./data/result.json", type=str,
                    help='The storage path for the prediction results')
# 选择标准答案数据集的路径
parser.add_argument('--gold_path', default="./data/gold_result.json", type=str,
                    help='The path to the standard answer dataset')
# 选择相似度计算模型的路径
parser.add_argument('--simModel_path', default=SimModel_path, type=str,
                    help='The similarity model used for calculating scores')
# 选择评测指标数据的存储路径
parser.add_argument('--metric_path', default="./data/metrics.json", type=str,
                    help='The storage path for evaluation metric data')
# 选择PDF文件处理后的缓存数据路径
parser.add_argument('--data_path', default="./all_text.txt", type=str,
                    help='The storage path after parsing the PDF file')
# 选择m3e召回的向量数据缓存路径
parser.add_argument('--m3e_vector_path', default="./vector_db/faiss_m3e_index", type=str,
                    help='The vector database based on M3E recall')
# 选择bge召回的向量数据缓存路径
parser.add_argument('--bge_vector_path', default="./vector_db/faiss_bge_index", type=str,
                    help='The vector database based on BGE recall')

args = parser.parse_args([])

In [4]:
from modelscope import snapshot_download
snapshot_download(repo_id="AI-ModelScope/m3e-large", local_dir="./pre_train_model/m3e-large")

2026-04-03 18:12:38,075 - modelscope - INFO - Got 8 files, start to download ...


Processing 8 items:   0%|          | 0.00/8.00 [00:00<?, ?it/s]

2026-04-03 18:14:00,360 - modelscope - INFO - Download model 'AI-ModelScope/m3e-large' successfully.


'./pre_train_model/m3e-large'

In [5]:
snapshot_download(repo_id="BAAI/bge-m3", local_dir="./pre_train_model/bge-m3")

2026-04-03 18:14:10,848 - modelscope - INFO - Got 29 files, start to download ...


Processing 29 items:   0%|          | 0.00/29.0 [00:00<?, ?it/s]

2026-04-03 18:17:37,572 - modelscope - INFO - Download model 'BAAI/bge-m3' successfully.


'./pre_train_model/bge-m3'

In [6]:
snapshot_download(repo_id="maple77/bce-reranker-base_v1", local_dir="./pre_train_model/bce-reranker-base_v1")

2026-04-03 18:17:57,427 - modelscope - INFO - Got 8 files, start to download ...


Processing 8 items:   0%|          | 0.00/8.00 [00:00<?, ?it/s]

2026-04-03 18:18:51,072 - modelscope - INFO - Download model 'maple77/bce-reranker-base_v1' successfully.


'./pre_train_model/bce-reranker-base_v1'

In [7]:
snapshot_download(repo_id="thomas/text2vec-base-chinese", local_dir="./pre_train_model/text2vec-base-chinese")

2026-04-03 18:18:56,539 - modelscope - INFO - Got 9 files, start to download ...


Processing 9 items:   0%|          | 0.00/9.00 [00:00<?, ?it/s]

2026-04-03 18:19:19,184 - modelscope - INFO - Download model 'thomas/text2vec-base-chinese' successfully.


'./pre_train_model/text2vec-base-chinese'

In [4]:
from pdf_parse import DataProcess

# 初始化 PDF 解析器
dp = DataProcess("./data/car_user_manual.pdf")

# 执行多种解析策略（与 retriever 中相同）
dp.ParseBlock(max_seq=1024)
dp.ParseBlock(max_seq=512)
dp.ParseAllPage(max_seq=256)
dp.ParseAllPage(max_seq=512)
dp.ParseOnePageWithRule(max_seq=256)
dp.ParseOnePageWithRule(max_seq=512)

print("PDF 解析完成，开始写入 all_text.txt")

# 将解析结果写入 all_text.txt
with open("./all_text.txt", "w", encoding="utf-8") as f:
    for line in dp.data:
        f.write(line + "\n")

print("all_text.txt 生成完成！")

PDF 解析完成，开始写入 all_text.txt
all_text.txt 生成完成！


In [ ]:
question_test(
        model_name=args.llm_name,
        reranker_name=args.reranker_name,
        m3e_embeddings_model_path=args.m3e_embeddings_model,
        bge_embeddings_model_path=args.bge_embeddings_model,
        test_path=args.test_path,
        output_path=args.predict_path,
        prompt_enhance=args.prompt_enhance,
        single_max_length=args.single_max_length,
        single_top_k=args.single_top_k,
        mutil_max_length=args.mutil_max_length,
        mutil_top_k=args.mutil_top_k,
        pdf_path=args.pdf_path,
        data_path=args.data_path,                 # pdf文件处理后的缓存数据
        #第一次运行需要注释，之后可以取消注释以使用缓存的向量数据
        #m3e_vector_path=args.m3e_vector_path,     # m3e召回的向量数据缓存
        #bge_vector_path=args.bge_vector_path,     # bge召回的向量数据缓存
    )

No sentence-transformers model found with name ./pre_train_model/m3e-large. Creating a new one with mean pooling.


m3e_retriever load ok
bge_retriever load ok
bm25 load ok
tf-idf load ok


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

llm load ok
rerank model load ok
103


  0%|          | 0/103 [00:00<?, ?it/s]The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
  1%|          | 1/103 [00:02<03:29,  2.05s/it]The following generation flags are not valid and may be 

cost time: 20.683333333333334minutes


In [6]:
test_metrics(
        gold_path=args.gold_path,
        predict_path=args.predict_path,
        metric_path=args.metric_path,
        simModel_path=args.simModel_path,
    )

Read gold from ./data/gold_result.json
Read predict file from ./data/result.json
预测: 中国足球的队长是谁, 得分: 1.0
预测: 新冠肺炎如何预防？, 得分: 1.0
预测: 交通事故如何处理？, 得分: 0.379097580909729
预测: 怎样加热座椅？, 得分: 0.9510444402694702
预测: 自动模式下，中央显示屏是如何切换日间和夜间模式的？, 得分: 0.9543520510196686
预测: 如何通过中央显示屏进行副驾驶员座椅设置？, 得分: 0.950123518705368
预测: 副仪表台按钮如何操作中央显示屏？, 得分: 0.987622857093811
预测: 如何从锁定状态唤醒中央显示器?, 得分: 0.8134288787841797
预测: 如何正确使用颈椎保护系统？, 得分: 0.956772118806839
预测: 前方交叉路口预警系统（FCTA）的作用是什么？, 得分: 0.9964123368263245
预测: 在使用FCTA时需要注意哪些事项？, 得分: 0.9744159877300262
预测: 如何打开车辆尾门？, 得分: 0.9011433720588684
预测: 在哪些情况下智能钥匙可能会受到干扰，导致功能异常？, 得分: 0.9809519648551941
预测: 车辆尾门的防夹保护功能是如何工作的？, 得分: 0.9804088473320007
预测: 在操作电动后备厢时需要注意哪些事项？, 得分: 0.9283533096313477
预测: 如何进入车辆功能界面？, 得分: 0.9567942023277283
预测: 在车辆功能界面有哪些操作选项？, 得分: 0.9376606643199921
预测: 如何编辑快捷开关图标?, 得分: 0.9884528517723083
预测: 如何减少车辆腐蚀风险？, 得分: 0.9835796356201172
预测: 如何通过空调系统面板调节空调风量？, 得分: 0.9260261654853821
预测: 如何创建新的Lynk&CoID？, 得分: 0.9670853912830353
预测: 什么是车主账户？, 得分: 0.9678490757